# Example 7.11.1 / 2: Pattern Formulation

Companion notebook for **7.11.1 Bin Packing and Cutting** in the AMPL lab Documentation, section "Pattern Formulation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Pattern Formulation

For cutting stock, let each pattern `p` cut `a[i,p]` pieces of item `i` from one roll, and choose
an integer number `UsePattern[p]` of rolls cut with it. Here rolls are 10 wide, and three
patterns cut widths 3, 4 and 5:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                        # widths ordered by customers
set PATTERNS;                     # ways to cut one roll
param width {ITEMS} > 0;          # width of each item
param demand {ITEMS} >= 0;        # pieces ordered
param roll_width > 0;             # width of a stock roll
param a {ITEMS, PATTERNS} integer >= 0;  # pieces of the item in the pattern
check {p in PATTERNS}: sum {i in ITEMS} width[i] * a[i,p] <= roll_width;  # pattern fits
var UsePattern {PATTERNS} integer >= 0;  # rolls cut with the pattern

minimize Rolls: sum {p in PATTERNS} UsePattern[p];
# chosen patterns cover the demand for each item
subject to MeetDemand {i in ITEMS}:
    sum {p in PATTERNS} a[i,p] * UsePattern[p] >= demand[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["w3", "w4", "w5"]
ampl.set["PATTERNS"] = ["p1", "p2", "p3"]
ampl.param["width"] = {"w3": 3, "w4": 4, "w5": 5}
ampl.param["demand"] = {"w3": 9, "w4": 5, "w5": 4}
ampl.param["roll_width"] = 10
ampl.param["a"] = {("w3", "p1"): 3, ("w4", "p1"): 0, ("w5", "p1"): 0,
                   ("w3", "p2"): 1, ("w4", "p2"): 1, ("w5", "p2"): 0,
                   ("w3", "p3"): 0, ("w4", "p3"): 0, ("w5", "p3"): 2}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.var["UsePattern"].to_dict())

HiGHS 1.14.0: optimal solution; objective 9
0 simplex iterations
0 branching nodes
{'p1': 2, 'p2': 5, 'p3': 2}


### Pure AMPL: data.dat and model.run

The `check` statement verifies, when the data is read, that each pattern fits in one roll.
Nine rolls meet the demand, with two extra pieces of width 3: `>= demand[i]` permits
overproduction; replace it with equality only when surplus is forbidden and the available
patterns can meet demand exactly.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: ITEMS: width demand := w3 3 9  w4 4 5  w5 5 4;
param roll_width := 10;
param a:  p1 p2 p3 :=
     w3    3  1  0
     w4    0  1  0
     w5    0  0  2 ;
set PATTERNS := p1 p2 p3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display UsePattern;

HiGHS 1.14.0: optimal solution; objective 9
0 simplex iterations
0 branching nodes
UsePattern [*] :=
p1  2
p2  5
p3  2
;

